# 🔬 Detector de sedimento urinario — entrenamiento en Google Colab

Entrena un modelo **YOLO11** que detecta 7 estructuras del sedimento urinario: eritrocito, leucocito, célula epitelial, núcleo epitelial, cilindro, cristal y levadura. Al final genera el archivo `.onnx` para la app del celular.

| Fuente | Qué aporta | Cómo llega |
|---|---|---|
| [Urinary Sediment Dataset](https://github.com/174614361/Urinary-Sediment-Dataset) (Liang et al.) | 5,376 fotos, 7 clases. Es la base principal | Google Drive (paso 3) |
| [UMID](https://github.com/dipamgoswami/UMID-Urine-Microscopic-Image-Dataset) | 366 fotos de microscopio manual: eritrocitos, leucocitos, epiteliales | Se descarga sola |
| Tu carpeta `mis_anotaciones` | Tus fotos ya etiquetadas (formato YOLO) | Opcional |

**Antes de empezar:** *Entorno de ejecución → Cambiar tipo de entorno de ejecución → T4 GPU*. Ejecuta las celdas en orden.

| Paso | Tiempo aprox. |
|---|---|
| Descargar y preparar datos | 5–15 min |
| Entrenar (40 épocas, GPU T4) | 1–2 h |

**Límites que conviene saber desde ahora**
- "Cilindro" y "Cristal" son clases generales: el modelo no distingue hialino de granuloso, ni oxalato de ácido úrico. Para eso hacen falta tus propias fotos etiquetadas.
- No detecta bacterias, espermatozoides, moco ni Trichomonas.
- La base principal viene de un analizador automático, no de fotos tomadas por el ocular. Con tus fotos rendirá menos hasta que agregues ejemplos propios.

> ⚠️ Herramienta educativa. No sustituye el examen de laboratorio.

## 1. Comprobar GPU e instalar librerías

In [ ]:
!nvidia-smi -L
!pip install -q ultralytics gdown

## 2. Conectar Google Drive (para guardar el modelo)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
PROYECTO = '/content/drive/MyDrive/detector_orina'
!mkdir -p {PROYECTO}

## 3. Conseguir el Urinary Sediment Dataset

La celda intenta descargarlo sola desde la carpeta pública de Google Drive de los autores.

**Si la descarga automática falla** (Google a veces la bloquea), hazlo a mano una sola vez:
1. Abre https://drive.google.com/drive/folders/18VqmoqK7dVSdxiyEE6qCfIpS8UOqqIBS
2. Descarga el contenido a tu PC (queda un `.zip` o `.rar`).
3. Súbelo a tu Drive, dentro de la carpeta `detector_orina`.
4. Vuelve a ejecutar esta celda: lo encuentra y lo descomprime.

Si tampoco lo consigues, puedes seguir: el modelo se entrenará solo con UMID (3 clases: eritrocito, leucocito, célula epitelial).

In [ ]:
import glob, os, shutil, subprocess
from pathlib import Path

VOC = Path('/content/use')
VOC.mkdir(exist_ok=True)

def hay_anotaciones():
    return any(VOC.rglob('*.xml'))

def descomprimir_todo(carpeta):
    for f in list(Path(carpeta).rglob('*')):
        if f.suffix.lower() in {'.zip', '.rar', '.7z', '.tar', '.gz', '.tgz'} and f.is_file():
            print('Descomprimiendo', f.name, '…')
            subprocess.run(['7z', 'x', '-y', '-bso0', '-bsp0', f'-o{VOC}', str(f)])

# a) ¿ya lo subiste a tu Drive?
if not hay_anotaciones():
    descomprimir_todo(PROYECTO)
    for carpeta in Path(PROYECTO).rglob('Annotations'):   # o subiste la carpeta ya descomprimida
        if any(carpeta.glob('*.xml')):
            VOC = carpeta.parent
            break

# b) descarga automática
if not hay_anotaciones():
    print('Intentando la descarga automática…')
    subprocess.run(['gdown', '--folder', '--remaining-ok', '-O', '/content/use_descarga',
                    'https://drive.google.com/drive/folders/18VqmoqK7dVSdxiyEE6qCfIpS8UOqqIBS'])
    if os.path.isdir('/content/use_descarga'):
        descomprimir_todo('/content/use_descarga')
        if not hay_anotaciones() and any(Path('/content/use_descarga').rglob('*.xml')):
            VOC = Path('/content/use_descarga')

n = len(list(VOC.rglob('*.xml')))
if n:
    print(f'✅ Urinary Sediment Dataset listo: {n} archivos de anotación en {VOC}')
else:
    print('⚠️ No se consiguió el Urinary Sediment Dataset. Sigue las instrucciones de arriba para subirlo a mano,')
    print('   o continúa: se entrenará solo con UMID (3 clases).')

## 4. Descargar UMID y unir todo en un solo dataset

In [ ]:
%%writefile preparar_orina.py
"""
Prepara un dataset YOLO de sedimento urinario a partir de:
  1) Urinary Sediment Dataset (Liang et al., formato PASCAL VOC, 7 clases)
  2) UMID (Goswami et al., CSV, 3 clases)
  3) Tus propias fotos etiquetadas (formato YOLO), opcional

Uso:
    python preparar_orina.py --salida dataset_orina [--voc carpeta] [--umid carpeta] [--mias carpeta]
"""
import argparse
import csv
import random
import shutil
import statistics
import xml.etree.ElementTree as ET
from collections import Counter, defaultdict
from pathlib import Path

from PIL import Image

CLASES = ["Eritrocito", "Leucocito", "Célula epitelial", "Núcleo epitelial",
          "Cilindro", "Cristal", "Levadura"]

# nombre original -> clase unificada
MAPA = {
    "eryth": "Eritrocito", "rbc": "Eritrocito",
    "leuko": "Leucocito", "pus": "Leucocito", "wbc": "Leucocito",
    "epith": "Célula epitelial", "ep": "Célula epitelial",
    "epithn": "Núcleo epitelial",
    "cast": "Cilindro",
    "cryst": "Cristal",
    "mycete": "Levadura",
}
EXT = {".jpg", ".jpeg", ".png", ".bmp"}
AZAR = random.Random(42)


def guardar(salida: Path, origen: Path, nombre: str, split: str, cajas, copiar=False) -> bool:
    """cajas: lista de (clase_id, cx, cy, w, h) normalizados. Imágenes sin cajas no se guardan."""
    if not cajas:
        return False
    di, dl = salida / "images" / split, salida / "labels" / split
    di.mkdir(parents=True, exist_ok=True)
    dl.mkdir(parents=True, exist_ok=True)
    destino = di / nombre
    if not destino.exists():
        if copiar:
            shutil.copy2(origen, destino)
        else:
            try:
                destino.symlink_to(origen.resolve())
            except OSError:
                shutil.copy2(origen, destino)
    (dl / (Path(nombre).stem + ".txt")).write_text(
        "\n".join(f"{c} {cx:.6f} {cy:.6f} {w:.6f} {h:.6f}" for c, cx, cy, w, h in cajas))
    return True


def caja(x1, y1, x2, y2, W, H):
    x1, x2 = sorted((max(0.0, x1), min(float(W), x2)))
    y1, y2 = sorted((max(0.0, y1), min(float(H), y2)))
    w, h = x2 - x1, y2 - y1
    if w < 2 or h < 2:
        return None
    return (x1 + w / 2) / W, (y1 + h / 2) / H, w / W, h / H


# ---------- 1. Urinary Sediment Dataset (VOC) ----------
def sumar_voc(raiz: Path, salida: Path) -> int:
    xmls = [p for p in raiz.rglob("*.xml") if "__MACOSX" not in p.parts]
    if not xmls:
        print(f"  No hay archivos .xml (anotaciones VOC) en {raiz}")
        return 0
    imagenes = {}
    for p in raiz.rglob("*"):
        if p.suffix.lower() in EXT and "__MACOSX" not in p.parts:
            imagenes.setdefault(p.stem, p)
    # usa la partición original si existe (el test se reparte a validación)
    particion = {}
    for nombre, split in (("train", "train"), ("val", "val"), ("test", "val")):
        for txt in raiz.rglob(f"ImageSets/Main/{nombre}.txt"):
            for linea in txt.read_text().split():
                particion[linea.strip()] = split
    puestas, desconocidas = 0, Counter()
    for x in xmls:
        origen = imagenes.get(x.stem)
        if origen is None:
            continue
        try:
            arbol = ET.parse(x).getroot()
        except ET.ParseError:
            continue
        tam = arbol.find("size")
        W = int(float(tam.findtext("width", "0"))) if tam is not None else 0
        H = int(float(tam.findtext("height", "0"))) if tam is not None else 0
        if W < 2 or H < 2:
            W, H = Image.open(origen).size
        cajas = []
        for obj in arbol.iter("object"):
            nombre = (obj.findtext("name") or "").strip().lower()
            destino = MAPA.get(nombre)
            if destino is None:
                desconocidas[nombre] += 1
                continue
            b = obj.find("bndbox")
            if b is None:
                continue
            c = caja(*(float(b.findtext(k, "0")) for k in ("xmin", "ymin", "xmax", "ymax")), W, H)
            if c:
                cajas.append((CLASES.index(destino), *c))
        split = particion.get(x.stem) or ("val" if AZAR.random() < 0.15 else "train")
        if guardar(salida, origen, "use_" + origen.name, split, cajas):
            puestas += 1
    if desconocidas:
        print("  Etiquetas ignoradas (no reconocidas):", dict(desconocidas))
    print(f"  Urinary Sediment Dataset: {puestas} fotos.")
    return puestas


# ---------- 2. UMID (CSV) ----------
def sumar_umid(repo: Path, salida: Path) -> int:
    filas = []
    for archivo, split in (("train.csv", "train"), ("val.csv", "val"), ("test.csv", "val")):
        ruta = next(iter(repo.rglob(archivo)), None)
        if ruta is None:
            continue
        with ruta.open(newline="") as f:
            for r in csv.DictReader(f):
                r["split"] = split
                filas.append(r)
    if not filas:
        print(f"  No se encontraron train.csv / val.csv en {repo}")
        return 0
    imagenes = {p.name: p for p in repo.rglob("*") if p.suffix.lower() in EXT}
    # Algunas células en cúmulos están marcadas con un punto, no con recuadro:
    # se les da el tamaño mediano de su clase.
    medianas = {}
    for clase in ("pus", "rbc", "ep"):
        ws = [float(r["xmax"]) - float(r["xmin"]) for r in filas if r["label"] == clase]
        hs = [float(r["ymax"]) - float(r["ymin"]) for r in filas if r["label"] == clase]
        if ws:
            medianas[clase] = (statistics.median(ws), statistics.median(hs))
    por_imagen = defaultdict(list)
    for r in filas:
        por_imagen[r["image"]].append(r)
    puestas = 0
    for nombre, lista in por_imagen.items():
        origen = imagenes.get(Path(nombre).name)
        if origen is None:
            continue
        W, H = Image.open(origen).size
        cajas = []
        for r in lista:
            etiqueta = r["label"].strip().lower()
            x1, y1, x2, y2 = (float(r[k]) for k in ("xmin", "ymin", "xmax", "ymax"))
            if etiqueta.startswith("point-"):
                etiqueta = etiqueta[6:]
                if etiqueta not in medianas:
                    continue
                mw, mh = medianas[etiqueta]
                x1, x2, y1, y2 = x1 - mw / 2, x1 + mw / 2, y1 - mh / 2, y1 + mh / 2
            destino = MAPA.get(etiqueta)
            if destino is None:  # 'missedlabel' = estructura dudosa: se deja como fondo
                continue
            c = caja(x1, y1, x2, y2, W, H)
            if c:
                cajas.append((CLASES.index(destino), *c))
        if guardar(salida, origen, "umid_" + origen.name, lista[0]["split"], cajas):
            puestas += 1
    print(f"  UMID: {puestas} fotos.")
    return puestas


# ---------- 3. Fotos propias (YOLO) ----------
def sumar_mias(base: Path, salida: Path, nombres: list) -> int:
    clases_txt = base / "classes.txt"
    if not clases_txt.exists():
        print("  Sin mis_anotaciones/classes.txt: no hay fotos propias que sumar.")
        return 0
    ids = []
    for linea in clases_txt.read_text(encoding="utf-8").splitlines():
        n = linea.strip()
        if not n:
            continue
        n = MAPA.get(n.lower(), n)
        igual = next((x for x in nombres if x.lower() == n.lower()), None)
        if igual is None:
            nombres.append(n)
            igual = n
        ids.append(nombres.index(igual))
    puestas = 0
    for origen in (base / "images").rglob("*"):
        if origen.suffix.lower() not in EXT:
            continue
        txt = base / "labels" / (origen.stem + ".txt")
        if not txt.exists():
            continue
        cajas = []
        for linea in txt.read_text().splitlines():
            p = linea.split()
            if len(p) == 5 and 0 <= int(p[0]) < len(ids):
                cajas.append((ids[int(p[0])], *map(float, p[1:])))
        split = "val" if AZAR.random() < 0.15 else "train"
        if guardar(salida, origen, "mia_" + origen.name, split, cajas):
            puestas += 1
    print(f"  Fotos propias: {puestas}.")
    return puestas


def main():
    ap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    ap.add_argument("--salida", type=Path, default=Path("dataset_orina"))
    ap.add_argument("--voc", type=Path)
    ap.add_argument("--umid", type=Path)
    ap.add_argument("--mias", type=Path)
    a = ap.parse_args()

    nombres = list(CLASES)
    total = 0
    for etiqueta, ruta, paso in (("Urinary Sediment Dataset", a.voc, lambda r: sumar_voc(r, a.salida)),
                                 ("UMID", a.umid, lambda r: sumar_umid(r, a.salida)),
                                 ("fotos propias", a.mias, lambda r: sumar_mias(r, a.salida, nombres))):
        if ruta is None or not ruta.exists():
            print(f"- {etiqueta}: no disponible, se omite.")
            continue
        print(f"- {etiqueta}:")
        try:
            total += paso(ruta)
        except Exception as e:  # una fuente con problemas no detiene el resto
            print(f"  ⚠️ No se pudo usar: {type(e).__name__}: {e}")
    if total == 0:
        raise SystemExit("No se preparó ninguna foto. Revisa los pasos de descarga.")

    yaml = [f"path: {a.salida.resolve()}", "train: images/train", "val: images/val", "names:"]
    yaml += [f'  {i}: "{n}"' for i, n in enumerate(nombres)]
    (a.salida / "data.yaml").write_text("\n".join(yaml) + "\n", encoding="utf-8")

    cuenta = {"train": Counter(), "val": Counter()}
    for split in cuenta:
        for t in (a.salida / "labels" / split).glob("*.txt"):
            for linea in t.read_text().splitlines():
                if linea.strip():
                    cuenta[split][int(linea.split()[0])] += 1
    print(f"\nTotal: {total} fotos. Marcas por clase (entrenamiento / validación):")
    for i, n in enumerate(nombres):
        print(f"  {n:<20} {cuenta['train'][i]:>6} / {cuenta['val'][i]}")
    print("Configuración:", a.salida / "data.yaml")


if __name__ == "__main__":
    main()


In [ ]:
!rm -rf /content/umid /content/dataset_orina
!git clone -q --depth 1 https://github.com/dipamgoswami/UMID-Urine-Microscopic-Image-Dataset /content/umid
!python preparar_orina.py --salida /content/dataset_orina --voc "{VOC}" --umid /content/umid --mias "{PROYECTO}/mis_anotaciones"

Revisa la lista de **marcas por clase**. Con el dataset principal debe haber miles de eritrocitos y cientos o miles del resto. Si Cilindro, Cristal y Levadura salen en 0, el paso 3 no funcionó.

## 5. Entrenar
`yolo11s` es un poco más grande que el de parásitos porque las células son pequeñas. Si quieres ir más rápido, cambia a `yolo11n.pt`.

In [ ]:
from ultralytics import YOLO
modelo = YOLO('yolo11s.pt')
res = modelo.train(
    data='/content/dataset_orina/data.yaml',
    epochs=40, imgsz=640, batch=16,
    project=PROYECTO, name='entrenamiento', exist_ok=True,
    # aumentos para parecerse a fotos tomadas con celular por el ocular:
    hsv_h=0.02, hsv_s=0.5, hsv_v=0.4, degrees=180, flipud=0.5, scale=0.4,
)
MEJOR = f'{res.save_dir}/weights/best.pt'
print('Modelo guardado en:', MEJOR)

### 5b. (Solo si Colab se desconectó) Reanudar el entrenamiento
Repite antes los pasos 1 a 4 y luego ejecuta esta celda en lugar de la 5.

In [ ]:
from ultralytics import YOLO
YOLO(f'{PROYECTO}/entrenamiento/weights/last.pt').train(resume=True)
MEJOR = f'{PROYECTO}/entrenamiento/weights/best.pt'

## 6. Validar

- **mAP50** general: los autores de la base reportan alrededor de 0.84 con sus mejores modelos; entre 0.75 y 0.85 es un buen resultado.
- Por clase, **mAP50 menor a 0.50** significa que esa estructura todavía no es confiable.

In [ ]:
from pathlib import Path
from IPython.display import Image as Img, display
from ultralytics import YOLO

MEJOR = f"{PROYECTO}/entrenamiento/weights/best.pt"
m = YOLO(MEJOR).val(data="/content/dataset_orina/data.yaml", split="val", imgsz=640, plots=True)
print(f"mAP50:    {m.box.map50:.3f}")
print(f"mAP50-95: {m.box.map:.3f}\n")
print(f"{'clase':<22} {'mAP50':>8} {'mAP50-95':>10}")
lineas = [f"mAP50\t{m.box.map50:.3f}", f"mAP50-95\t{m.box.map:.3f}", "", "clase\tmAP50\tmAP50-95"]
for pos, i in enumerate(m.box.ap_class_index):
    ap50, ap = float(m.box.ap50[pos]), float(m.box.ap[pos])
    print(f"{m.names[int(i)]:<22} {ap50:8.3f} {ap:10.3f}{'   no confiable todavía' if ap50 < 0.5 else ''}")
    lineas.append(f"{m.names[int(i)]}\t{ap50:.3f}\t{ap:.3f}")
(Path(PROYECTO) / "entrenamiento" / "map.txt").write_text("\n".join(lineas) + "\n")
matriz = Path(m.save_dir) / "confusion_matrix_normalized.png"
if matriz.exists():
    display(Img(str(matriz), width=700))

## 7. Probar con tus propias fotos

In [ ]:
from google.colab import files
from IPython.display import Image as Img, display
from ultralytics import YOLO
import cv2
subidas = files.upload()
for nombre in subidas:
    r = YOLO(MEJOR).predict(nombre, conf=0.25)[0]
    display(Img(data=cv2.imencode('.jpg', r.plot())[1].tobytes(), width=600))
    cuenta = {}
    for c in r.boxes:
        cuenta[r.names[int(c.cls)]] = cuenta.get(r.names[int(c.cls)], 0) + 1
    print(nombre, '→', cuenta or 'sin detecciones')

## 8. Exportar para la app del celular
Genera `modelo_orina.onnx` en tu Drive: `detector_orina/modelo_orina.onnx`. Ese archivo se carga en la app con el botón **Cargar modelo**.

In [ ]:
import shutil
from ultralytics import YOLO
MEJOR = f'{PROYECTO}/entrenamiento/weights/best.pt'
salida = YOLO(MEJOR).export(format='onnx', imgsz=640, opset=17, simplify=True)
shutil.copy(salida, f'{PROYECTO}/modelo_orina.onnx')
print('Listo:', f'{PROYECTO}/modelo_orina.onnx')